In [1]:
!pip install openai anthropic

In [2]:
import os
import getpass

def _clean_key(raw):
    """Strip non-ASCII characters that can sneak in from copy-paste."""
    return ''.join(c for c in raw if ord(c) < 128).strip()

os.environ["ANTHROPIC_API_KEY"] = _clean_key(getpass.getpass("Anthropic API Key: "))
os.environ["XAI_API_KEY"]       = _clean_key(getpass.getpass("xAI API Key: "))

print("Keys set.")

Anthropic API Key:  ········
xAI API Key:  ········


Keys set.


In [3]:
import anthropic
from openai import OpenAI

anthropic_client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

grok_client = OpenAI(
    api_key=os.environ["XAI_API_KEY"],
    base_url="https://api.x.ai/v1"
)

print("All clients initialized.")

All clients initialized.


In [4]:
def ask_claude(prompt):
    response = anthropic_client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=500,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text.strip()


def ask_grok(prompt):
    response = grok_client.chat.completions.create(
        model="grok-3-mini",
        max_tokens=500,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content.strip()


# Smoke test
print("Claude:", ask_claude("Say hello in one sentence."))
print("Grok:  ", ask_grok("Say hello in one sentence."))

Claude: Hello! I'm happy to help you today.
Grok:   Hello!


In [5]:
def build_queries(brand, category, product, competitor=None):
    queries = [
        {
            "type": "awareness",
            "prompt": f"What is {brand}? Describe what they do in {category} in 2-3 sentences."
        },
        {
            "type": "recommendation",
            "prompt": f"What are the best {product} brands for {category}? List 5 specific brand names and why you recommend each."
        },
        {
            "type": "evaluation",
            "prompt": f"Is {brand} a good {product} for {category}? What are their strengths and weaknesses?"
        },
    ]
    if competitor:
        queries.append({
            "type": "comparison",
            "prompt": f"{brand} vs {competitor} as a {product} for {category}: which would you recommend and why?"
        })
    return queries

In [6]:
def run_audit(brand, category, product, competitor=None):
    queries = build_queries(brand, category, product, competitor)
    results = []
    for q in queries:
        print(f"  Querying [{q['type']}]...")
        results.append({
            "type":   q["type"],
            "prompt": q["prompt"],
            "claude": ask_claude(q["prompt"]),
            "grok":   ask_grok(q["prompt"])
        })
    return results

In [7]:
def check_accuracy_with_judge(brand, evaluation_text):
    """Use Claude-as-judge to assess whether the evaluation is actually informative."""
    judge_prompt = (
        f"Does this response give specific, accurate information about {brand}?\n"
        "Answer with ONLY one word: 'accurate', 'vague', or 'wrong'\n\n"
        f"Response: {evaluation_text}"
    )
    result = ask_claude(judge_prompt).strip().lower()
    return result == "accurate"


def check_wins_comparison(brand, comparison_text):
    """Use Claude-as-judge to determine who won the comparison."""
    judge_prompt = (
        "Read this product comparison response carefully.\n"
        "Who does the response recommend more strongly?\n"
        f"Answer with ONLY one word: '{brand}', 'competitor', or 'tie'.\n\n"
        f"Response: {comparison_text}"
    )
    result = ask_claude(judge_prompt)
    return brand.lower() in result.lower()


def score_model(brand, results, model_key):
    brand_lower = brand.lower()
    report = {}

    for r in results:
        response = r[model_key]
        if r["type"] == "recommendation":
            report["recommendation"] = brand_lower in response.lower()
        elif r["type"] == "evaluation":
            report["evaluation"] = check_accuracy_with_judge(brand, response)
        elif r["type"] == "comparison":
            report["comparison"] = check_wins_comparison(brand, response)

    visibility = "HIGH" if report.get("recommendation") else "LOW"
    accuracy   = "HIGH" if report.get("evaluation")     else "LOW"

    quadrant_map = {
        ("HIGH", "HIGH"): "GEO Ready",
        ("HIGH", "LOW"):  "Visible but Wrong",
        ("LOW",  "HIGH"): "Accurate but Missing",
        ("LOW",  "LOW"):  "Invisible & Misrepresented",
    }

    return {
        "model":           model_key,
        "unprompted":      report.get("recommendation", False),
        "accurate":        report.get("evaluation",     False),
        "wins_comparison": report.get("comparison",     False),
        "visibility":      visibility,
        "accuracy":        accuracy,
        "diagnosis":       quadrant_map[(visibility, accuracy)]
    }


def score_audit(brand, results):
    c = score_model(brand, results, "claude")
    g = score_model(brand, results, "grok")

    print(f"\nAI BRAND FOOTPRINT: {brand.upper()}")
    print("=" * 58)
    print(f"  {'Metric':<24} {'Claude':^14} {'Grok':^14}")
    print("-" * 58)

    metrics = [
        ("Unprompted mention",  "unprompted"),
        ("Accurate evaluation", "accurate"),
        ("Wins comparison",     "wins_comparison"),
    ]
    for label, key in metrics:
        cv = "YES" if c[key] else "NO"
        gv = "YES" if g[key] else "NO"
        print(f"  {label:<24} {cv:^14} {gv:^14}")

    print("-" * 58)
    print(f"  {'VISIBILITY':<24} {c['visibility']:^14} {g['visibility']:^14}")
    print(f"  {'ACCURACY':<24} {c['accuracy']:^14} {g['accuracy']:^14}")
    print(f"  {'DIAGNOSIS':<24} {c['diagnosis']:^14} {g['diagnosis']:^14}")
    print("=" * 58)

    return {"claude": c, "grok": g}

In [8]:
results_cerave = run_audit(
    brand="CeraVe",
    category="sensitive skin",
    product="moisturizer",
    competitor="La Roche-Posay"
)

for r in results_cerave:
    print(f"\n[{r['type'].upper()}]")
    print(f"Claude: {r['claude']}")
    print(f"Grok:   {r['grok']}")

  Querying [awareness]...
  Querying [recommendation]...
  Querying [evaluation]...
  Querying [comparison]...

[AWARENESS]
Claude: # CeraVe

CeraVe is a dermatologist-developed skincare brand that specializes in products formulated with ceramides and hyaluronic acid to support the skin barrier. For sensitive skin, CeraVe products help restore and maintain the protective barrier, reducing irritation and inflammation while providing gentle hydration. Their fragrance-free formulations are designed to be non-irritating and suitable for easily reactive or compromised skin conditions.
Grok:   CeraVe is a dermatologist-recommended skincare brand owned by L'Oréal, specializing in gentle, science-backed products designed to restore and maintain the skin's natural barrier using key ingredients like ceramides, hyaluronic acid, and niacinamide.

For sensitive skin, CeraVe products work by replenishing essential ceramides to strengthen the skin's protective barrier, which helps reduce irritation, 

In [9]:
score_audit("CeraVe", results_cerave)


AI BRAND FOOTPRINT: CERAVE
  Metric                       Claude          Grok     
----------------------------------------------------------
  Unprompted mention            YES            YES      
  Accurate evaluation           YES            YES      
  Wins comparison               YES            YES      
----------------------------------------------------------
  VISIBILITY                    HIGH           HIGH     
  ACCURACY                      HIGH           HIGH     
  DIAGNOSIS                  GEO Ready      GEO Ready   


{'claude': {'model': 'claude',
  'unprompted': True,
  'accurate': True,
  'wins_comparison': True,
  'visibility': 'HIGH',
  'accuracy': 'HIGH',
  'diagnosis': 'GEO Ready'},
 'grok': {'model': 'grok',
  'unprompted': True,
  'accurate': True,
  'wins_comparison': True,
  'visibility': 'HIGH',
  'accuracy': 'HIGH',
  'diagnosis': 'GEO Ready'}}

In [10]:
results_dralthea = run_audit(
    brand="Dr. Althea",
    category="sensitive skin",
    product="moisturizer",
    competitor="CeraVe"
)

for r in results_dralthea:
    print(f"\n[{r['type'].upper()}]")
    print(f"Claude: {r['claude']}")
    print(f"Grok:   {r['grok']}")

  Querying [awareness]...
  Querying [recommendation]...
  Querying [evaluation]...
  Querying [comparison]...

[AWARENESS]
Claude: I don't have specific information about "Dr. Althea" in my training data. This could refer to a skincare brand, dermatologist, or product line, but I'm not certain which one you're asking about.

To give you accurate information about their approach to sensitive skin, could you provide more context? For example, is it a skincare brand, a clinic, or something else? That would help me give you a better answer.
Grok:   Dr. Althea is a skincare brand specializing in dermatologist-approved products designed for sensitive, acne-prone, and troubled skin, originating from South Korea and emphasizing clean, gentle formulations.

Their products work to soothe and protect sensitive skin by using non-irritating ingredients that hydrate, repair the skin barrier, and reduce inflammation, making them ideal for those with conditions like eczema or rosacea.

This approach 

In [11]:
score_audit("Dr. Althea", results_dralthea)


AI BRAND FOOTPRINT: DR. ALTHEA
  Metric                       Claude          Grok     
----------------------------------------------------------
  Unprompted mention             NO             NO      
  Accurate evaluation            NO             NO      
  Wins comparison                NO             NO      
----------------------------------------------------------
  VISIBILITY                    LOW            LOW      
  ACCURACY                      LOW            LOW      
  DIAGNOSIS                Invisible & Misrepresented Invisible & Misrepresented


{'claude': {'model': 'claude',
  'unprompted': False,
  'accurate': False,
  'wins_comparison': False,
  'visibility': 'LOW',
  'accuracy': 'LOW',
  'diagnosis': 'Invisible & Misrepresented'},
 'grok': {'model': 'grok',
  'unprompted': False,
  'accurate': False,
  'wins_comparison': False,
  'visibility': 'LOW',
  'accuracy': 'LOW',
  'diagnosis': 'Invisible & Misrepresented'}}